####역학(Epidemic) 피처 생성
#####- infected_farm_count_3km      : 기준일 이전 반경 3km 내 확진 농장 수 (유효기간 N일)
#####- outbreak_count_5yr           : 기준일 기준 최근 5년 반경 5km 내 발생 횟수

#####확진 소스     = 동일 파일 내 label_infected=1 행

In [ ]:
import sys
sys.path.append("/Workspace/방역로/00_Shared_Utils")
from utils_config import CATALOG

from pyspark.sql import functions as F, DataFrame
from pyspark.sql.types import DoubleType

In [ ]:
# 출력 테이블 
OUTPUT_TABLE = f"{CATALOG}.gold.farm_epidemic_features_daily"

In [ ]:
# ──────────────────────────────────────────────
# 파라미터 (수정 포인트)
# ──────────────────────────────────────────────
INFECTION_WINDOW_DAYS = 21   # infected_farm_count_3km 유효 기간 (일) — 팀 확정 필요
RADIUS_3KM            = 3.0  # infected_farm_count_3km 반경 (km) — 팀 확정 필요
RADIUS_5YR_KM         = 5.0  # outbreak_count_5yr 반경 (km)
YEARS_5               = 5    # outbreak_count_5yr 기간 (년)
 

In [ ]:
# 농장 데이터

# 카탈로그
FARM_FILE = spark.read.table(f"{CATALOG}.silver.farm_master")

In [ ]:
 
# ──────────────────────────────────────────────
# Haversine UDF (단위: km)
# ──────────────────────────────────────────────
@F.udf(DoubleType())
def haversine(lat1, lon1, lat2, lon2):
    from math import radians, sin, cos, sqrt, atan2
    if None in (lat1, lon1, lat2, lon2):
        return None
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

In [ ]:
# ──────────────────────────────────────────────
# 피처 계산 함수
# infection을 함수 안에서 로드 — 전역 DataFrame 클로저 참조 시 NameError 방지
# ──────────────────────────────────────────────

def build_epidemic_features() -> DataFrame:
    raw = FARM_FILE.select(
        "farm_id", "farm_name",
        F.col("latitude").alias("lat"),
        F.col("longitude").alias("lon"),
        "reference_date", "outbreak_date", "label_infected",
    )
    
    # 전체 농장 기준 (reference_date 7개 중복 → farm_id + reference_date 기준 dedup)
    farm = raw.dropDuplicates(["farm_id", "reference_date"])
 
    # 확진 소스: label_infected=1, outbreak_date 기준 dedup (reference_date 7개 중복 제거)
    infection = (
        raw
        .filter(F.col("label_infected") == 1)
        .dropDuplicates(["farm_id", "outbreak_date"])
        .select(
            F.col("lat").alias("inf_lat"),
            F.col("lon").alias("inf_lon"),
            F.col("outbreak_date").alias("inf_outbreak_date"),
        )
    )
 
    # farm × infection 크로스 조인 후 거리 계산
    crossed = farm.crossJoin(infection).withColumn(
        "dist_km",
        haversine(F.col("lat"), F.col("lon"), F.col("inf_lat"), F.col("inf_lon"))
    )
 
    # 피처 1. infected_farm_count_3km
    # ※ 유효 기간 변경 → INFECTION_WINDOW_DAYS, 반경 변경 → RADIUS_3KM
    feat1 = (
        crossed
        .filter(
            (F.col("dist_km") <= RADIUS_3KM) &
            (F.col("inf_outbreak_date") <= F.col("reference_date")) &
            (F.datediff(F.col("reference_date"), F.col("inf_outbreak_date")) <= INFECTION_WINDOW_DAYS)
        )
        .groupBy("farm_id", "reference_date")
        .agg(F.count("*").alias("infected_farm_count_3km"))
    )
 
    # 피처 2. outbreak_count_5yr
    feat2 = (
        crossed
        .filter(
            (F.col("dist_km") <= RADIUS_5YR_KM) &
            (F.col("inf_outbreak_date") <= F.col("reference_date")) &
            (F.col("inf_outbreak_date") >= F.add_months(F.col("reference_date"), -YEARS_5 * 12))
        )
        .groupBy("farm_id", "reference_date")
        .agg(F.count("*").alias("outbreak_count_5yr"))
    )
 
    return (
        farm.select("farm_id", "farm_name", "reference_date")
        .join(feat1, ["farm_id", "reference_date"], "left")
        .join(feat2, ["farm_id", "reference_date"], "left")
        .fillna(0, subset=["infected_farm_count_3km", "outbreak_count_5yr"])
    )

In [ ]:
# ──────────────────────────────────────────────
# 실행 및 저장
# ──────────────────────────────────────────────
result = build_epidemic_features()
result.writeTo(OUTPUT_TABLE).using("delta").createOrReplace()
print("저장 완료")
 

In [ ]:
%sql
select * from dt4_team1_databricks.gold.farm_epidemic_features_daily